# SST Evaluation — BLIP-2 Baseline
**Third notebook in the SST vs. VLM paper series.**

Purpose: establish BLIP-2 as a second VLM baseline alongside LLaVA, giving reviewers
a richer comparison point for the SST approach.

What this notebook does:
- Loads **BLIP-2 OPT-2.7B** from HuggingFace and runs it on the same GQA questions used in `sst_eval_main.ipynb`
- Reports exact match, lenient accuracy, and per-image latency
- Produces a **3-way comparison table**: BLIP-2 vs LLaVA vs best SST method
- Saves all outputs to the shared `outputs/results/` directory

> Prerequisites: run `sst_eval_main.ipynb` first so the GQA checkpoint and LLaVA
> results CSVs exist. This notebook reads from those files.


In [ ]:
import sys, os, json, re, time, random
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
    from PIL import Image
    from transformers import Blip2Processor, Blip2ForConditionalGeneration
    print(f"torch {torch.__version__} | transformers loaded")
except ImportError as e:
    raise ImportError(
        f"Missing dependency: {e}\n"
        "Run: pip install torch torchvision transformers accelerate Pillow"
    )

try:
    import tiktoken
    TOKEN_ENCODER = tiktoken.get_encoding("cl100k_base")
except Exception:
    TOKEN_ENCODER = None

random.seed(42)
np.random.seed(42)


In [ ]:
# ── Device detection ──────────────────────────────────────────────────────
# Priority: CUDA > MPS (Apple Silicon) > CPU
# Note: float16 causes NaN outputs on MPS for some BLIP-2 layers.
#       We use float32 on MPS and float16 on CUDA.

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    DTYPE  = torch.float16
    print(f"Using CUDA — {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    DTYPE  = torch.float32   # float16 has known NaN issues on MPS with BLIP-2
    print("Using MPS (Apple Silicon) — dtype=float32 to avoid NaN outputs")
else:
    DEVICE = torch.device("cpu")
    DTYPE  = torch.float32
    print("Using CPU — inference will be slow")

print(f"Device: {DEVICE} | Dtype: {DTYPE}")


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────
PROJECT_ROOT  = Path("..").resolve()
DATA_DIR      = PROJECT_ROOT / "data"
GQA_RAW_DIR   = DATA_DIR / "gqa_raw"
IMAGE_DIR     = DATA_DIR / "gqa_images_subset"
RESULTS_DIR   = PROJECT_ROOT / "outputs" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Config ─────────────────────────────────────────────────────────────────
# BLIP-2 is slower than Mistral — default to 1,000 questions for a first run.
# Increase to match sst_eval_main sample size once you've verified it works.
MAX_SAMPLES      = 1000
CHECKPOINT_FREQ  = 25           # save more often — BLIP-2 inference is slow
PREFIX           = f"blip2_gqa_{MAX_SAMPLES}_eval"
CHECKPOINT       = RESULTS_DIR / f"{PREFIX}_checkpoint.csv"

# Paths to sst_eval_main outputs — used for the comparison cell
import glob
_gqa_summaries  = sorted(glob.glob(str(RESULTS_DIR / "gqa_*_sst_eval_summary.csv")))
_llava_results  = sorted(glob.glob(str(RESULTS_DIR / "gqa_*_sst_eval_llava_results.csv")))
GQA_SUMMARY_CSV = _gqa_summaries[-1]  if _gqa_summaries  else None
LLAVA_CSV       = _llava_results[-1]  if _llava_results  else None

# BLIP-2 HuggingFace model ID
# blip2-opt-2.7b  — ~5.5 GB in fp16, ~10 GB in fp32 (recommended for M2)
# blip2-opt-6.7b  — too large for 16 GB RAM
# blip2-flan-t5-xl — alternative; change MODEL_ID if preferred
MODEL_ID = "Salesforce/blip2-opt-2.7b"

print("Project root:", PROJECT_ROOT)
print("Results dir: ", RESULTS_DIR)
print(f"GQA summary: {GQA_SUMMARY_CSV or 'NOT FOUND — run sst_eval_main first'}")
print(f"LLaVA CSV:   {LLAVA_CSV       or 'NOT FOUND — run sst_eval_main first'}")
print(f"Model:       {MODEL_ID}")
print(f"Max samples: {MAX_SAMPLES}")


## Install dependencies

If any imports failed in Cell 1, run:

```bash
pip install torch torchvision torchaudio
pip install transformers accelerate Pillow
```

**Memory requirements for BLIP-2 OPT-2.7B:**

| Device | Dtype | VRAM / RAM needed |
|--------|-------|-------------------|
| CUDA   | fp16  | ~6 GB VRAM |
| MPS (M2) | fp32 | ~11 GB unified memory |
| CPU    | fp32  | ~11 GB RAM (slow) |

If you hit OOM on MPS, set `MAX_SAMPLES = 200` and restart the kernel before loading the model.


In [ ]:
# ── Load BLIP-2 ───────────────────────────────────────────────────────────
# First run downloads ~5-10 GB from HuggingFace Hub — subsequent runs use cache.

print(f"Loading {MODEL_ID} ...")
t0 = time.perf_counter()

processor = Blip2Processor.from_pretrained(MODEL_ID)
model     = Blip2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
).to(DEVICE)
model.eval()

elapsed = time.perf_counter() - t0
print(f"Model loaded in {elapsed:.1f}s")

# Smoke-test
try:
    _dummy_img    = Image.new("RGB", (224, 224), color=(128, 128, 128))
    _dummy_inputs = processor(images=_dummy_img,
                               text="Question: What color is this? Answer:",
                               return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        _dummy_out = model.generate(**_dummy_inputs, max_new_tokens=5)
    _dummy_ans = processor.decode(_dummy_out[0], skip_special_tokens=True).strip()
    print(f"Smoke-test passed — dummy answer: '{_dummy_ans}'")
except Exception as e:
    print(f"Smoke-test failed: {e}")


In [ ]:
# ── Load GQA samples ──────────────────────────────────────────────────────
# Loads question_ids from the sst_eval_main checkpoint so BLIP-2 is evaluated
# on the exact same questions — essential for a fair comparison.

if not GQA_SUMMARY_CSV:
    raise FileNotFoundError(
        "GQA summary CSV not found. Run sst_eval_main.ipynb first."
    )

# Load question IDs used in sst_eval_main (from its checkpoint)
_gqa_ckpts = sorted(glob.glob(str(RESULTS_DIR / "gqa_*_sst_eval_checkpoint.csv")))
if not _gqa_ckpts:
    raise FileNotFoundError("GQA checkpoint CSV not found. Run sst_eval_main first.")

gqa_ckpt_df = pd.read_csv(_gqa_ckpts[-1], low_memory=False)
gqa_qids    = set(gqa_ckpt_df["question_id"].astype(str).unique())
print(f"GQA checkpoint has {len(gqa_qids):,} unique question IDs")

# Load raw GQA questions + scene graphs
_q_files = sorted(GQA_RAW_DIR.glob("*questions*.json"))
if not _q_files:
    raise FileNotFoundError(f"No GQA question JSON found in {GQA_RAW_DIR}")

all_questions = {}
for qf in _q_files:
    with open(qf) as f:
        all_questions.update(json.load(f))
print(f"Loaded {len(all_questions):,} GQA questions from {len(_q_files)} file(s)")

_sg_files = sorted(GQA_RAW_DIR.glob("*sceneGraphs*.json")) +             sorted(GQA_RAW_DIR.glob("*scene_graphs*.json"))
all_scene_graphs = {}
for sgf in _sg_files:
    with open(sgf) as f:
        all_scene_graphs.update(json.load(f))
print(f"Loaded {len(all_scene_graphs):,} scene graphs")

# Build sample list — only questions that (a) sst_eval_main used,
# (b) have an image on disk, and (c) have a scene graph
all_cleaned = []
for qid, q in all_questions.items():
    if str(qid) not in gqa_qids:
        continue
    img_id   = q.get("imageId", "")
    img_path = IMAGE_DIR / f"{img_id}.jpg"
    if not img_path.exists():
        continue
    if img_id not in all_scene_graphs:
        continue
    all_cleaned.append({
        "question_id": str(qid),
        "image_id":    img_id,
        "image_path":  str(img_path),
        "question":    q["question"],
        "answer":      q["answer"],
    })

random.shuffle(all_cleaned)
all_cleaned = all_cleaned[:MAX_SAMPLES]
print(f"\nFinal sample: {len(all_cleaned)} questions (target: {MAX_SAMPLES})")


In [ ]:
# ── Helper functions (shared with sst_eval_main) ──────────────────────────

_ARTICLES     = {"a", "an", "the"}
_PERIOD_STRIP = re.compile(r"(?!<=\d)(\.)(?!\d)")
_COMMA_STRIP  = re.compile(r"(\d)(,)(\d)")
_PUNCT        = set('!"&\'()*+,-./:;<=?@[\\]^_`{|}~')

def normalize_answer(s: str) -> str:
    if not isinstance(s, str): s = str(s)
    s = s.lower().strip()
    s = _PERIOD_STRIP.sub("", s)
    s = _COMMA_STRIP.sub(r"\1\3", s)
    s = "".join(c for c in s if c not in _PUNCT)
    s = " ".join(w for w in s.split() if w not in _ARTICLES)
    return s.strip()

def is_lenient_correct(pred: str, gt: str) -> bool:
    return pred == gt or pred in gt or gt in pred

def count_tokens(text: str) -> int:
    if TOKEN_ENCODER:
        return len(TOKEN_ENCODER.encode(text))
    return len(text.split())

print("Helper functions defined.")


In [ ]:
# ── query_blip2 ───────────────────────────────────────────────────────────

def query_blip2(image_path: str, question: str,
                max_new_tokens: int = 20) -> tuple[str, float]:
    """
    Run BLIP-2 VQA on a single image+question pair.
    Returns (raw_answer, elapsed_seconds).

    Prompt format: 'Question: {question} Answer:'
    This is the standard VQA prompt for BLIP-2 OPT models.
    For Flan-T5 variants the same format also works.
    """
    image  = Image.open(image_path).convert("RGB")
    prompt = f"Question: {question} Answer:"

    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt",
    ).to(DEVICE, DTYPE)

    t0 = time.perf_counter()
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=1,          # greedy — fast + deterministic
            length_penalty=-1.0,  # encourage short answers
        )
    elapsed = time.perf_counter() - t0

    # Decode — strip the input prompt echoed back in some model versions
    raw = processor.decode(output_ids[0], skip_special_tokens=True).strip()
    # OPT models sometimes echo the prompt; keep only the answer part
    if "Answer:" in raw:
        raw = raw.split("Answer:")[-1].strip()

    return raw, elapsed

# Warm-up pass (first inference is slow due to compilation)
print("Warming up BLIP-2 ...")
_test_img = all_cleaned[0]["image_path"]
_ans, _t  = query_blip2(_test_img, "What is in the image?")
print(f"Warm-up done ({_t*1000:.0f} ms) — answer: '{_ans}'")


In [ ]:
# ── BLIP-2 evaluation loop ────────────────────────────────────────────────

if CHECKPOINT.exists():
    ckpt_df  = pd.read_csv(CHECKPOINT, low_memory=False)
    done_set = set(ckpt_df["question_id"].astype(str))
    print(f"Resuming from checkpoint: {len(ckpt_df):,} rows done")
else:
    ckpt_df  = pd.DataFrame()
    done_set = set()
    print("Starting fresh")

remaining = [s for s in all_cleaned if str(s["question_id"]) not in done_set]
print(f"Total: {len(all_cleaned)} | Done: {len(done_set)} | Remaining: {len(remaining)}")

new_rows = []
for i, sample in enumerate(remaining):
    qid = str(sample["question_id"])
    try:
        raw_pred, elapsed = query_blip2(sample["image_path"], sample["question"])
        pred_norm = normalize_answer(raw_pred)
        gt_norm   = normalize_answer(sample["answer"])
        new_rows.append({
            "question_id":     qid,
            "image_id":        sample["image_id"],
            "question":        sample["question"],
            "true_answer":     sample["answer"],
            "prediction":      raw_pred,
            "prediction_norm": pred_norm,
            "exact_correct":   int(pred_norm == gt_norm),
            "lenient_correct": int(is_lenient_correct(pred_norm, gt_norm)),
            "latency_s":       round(elapsed, 3),
        })
    except Exception as e:
        print(f"  ⚠️  {qid} failed: {e}")

    if new_rows and (i + 1) % CHECKPOINT_FREQ == 0:
        ckpt_df = pd.concat([ckpt_df, pd.DataFrame(new_rows)], ignore_index=True)
        ckpt_df.to_csv(CHECKPOINT, index=False)
        new_rows = []
        pct = (len(done_set) + i + 1) / len(all_cleaned) * 100
        avg_ms = ckpt_df["latency_s"].mean() * 1000 if not ckpt_df.empty else 0
        print(f"  {i+1}/{len(remaining)} | {pct:.1f}% | avg latency {avg_ms:.0f} ms")

if new_rows:
    ckpt_df = pd.concat([ckpt_df, pd.DataFrame(new_rows)], ignore_index=True)
    ckpt_df.to_csv(CHECKPOINT, index=False)

print(f"\nDone. {len(ckpt_df):,} rows → {CHECKPOINT}")


In [ ]:
# ── Summarize BLIP-2 results ──────────────────────────────────────────────
results_df = pd.read_csv(CHECKPOINT, low_memory=False)

blip2_summary = {
    "model":            "BLIP-2 OPT-2.7B",
    "n":                len(results_df),
    "exact_accuracy":   results_df["exact_correct"].mean(),
    "lenient_accuracy": results_df["lenient_correct"].mean(),
    "avg_latency_ms":   results_df["latency_s"].mean() * 1000,
    "visual_tokens":    32,   # BLIP-2 Q-Former compresses to 32 query tokens
}

print("── BLIP-2 Results ────────────────────────────────────────────────────────")
print(f"  n:                 {blip2_summary['n']:,}")
print(f"  Exact accuracy:    {blip2_summary['exact_accuracy']:.1%}")
print(f"  Lenient accuracy:  {blip2_summary['lenient_accuracy']:.1%}")
print(f"  Avg latency:       {blip2_summary['avg_latency_ms']:.0f} ms")
print(f"  Visual tokens:     {blip2_summary['visual_tokens']} (Q-Former output)")
print(f"  LLaVA visual tok:  576 (ViT-L/14 @ 336px)")
print(f"  Compression vs LLaVA: {576 / blip2_summary['visual_tokens']:.1f}×")

# Save
blip2_result_df = pd.DataFrame([blip2_summary])
blip2_result_df.to_csv(RESULTS_DIR / f"{PREFIX}_summary.csv", index=False)
print(f"\nSummary saved → {RESULTS_DIR}/{PREFIX}_summary.csv")


In [ ]:
# ── 3-way comparison: BLIP-2 vs LLaVA vs best SST ────────────────────────

comparison_rows = []

# ── BLIP-2 ────────────────────────────────────────────────────────────────
comparison_rows.append({
    "model":           "BLIP-2 OPT-2.7B",
    "exact_accuracy":  blip2_summary["exact_accuracy"],
    "lenient_accuracy":blip2_summary["lenient_accuracy"],
    "avg_latency_ms":  blip2_summary["avg_latency_ms"],
    "visual_tokens":   blip2_summary["visual_tokens"],
    "type":            "VLM baseline",
})

# ── LLaVA (from sst_eval_main) ────────────────────────────────────────────
if LLAVA_CSV and Path(LLAVA_CSV).exists():
    llava_df = pd.read_csv(LLAVA_CSV, low_memory=False)
    # Subset to same question_ids as BLIP-2 evaluation for fairness
    blip2_qids = set(results_df["question_id"].astype(str))
    llava_sub  = llava_df[llava_df["question_id"].astype(str).isin(blip2_qids)]
    if llava_sub.empty:
        # Fallback: use all LLaVA rows (different sample, note it)
        llava_sub = llava_df
        print("Note: LLaVA and BLIP-2 evaluated on different subsets")
    comparison_rows.append({
        "model":           "LLaVA-1.5 7B",
        "exact_accuracy":  llava_sub["exact"].mean(),
        "lenient_accuracy":llava_sub["lenient"].mean() if "lenient" in llava_sub else None,
        "avg_latency_ms":  llava_sub["latency_s"].mean() * 1000,
        "visual_tokens":   576,
        "type":            "VLM baseline",
    })
else:
    print("LLaVA CSV not found — skipping LLaVA row")

# ── Best SST method + question_only (from sst_eval_main) ──────────────────
if GQA_SUMMARY_CSV and Path(GQA_SUMMARY_CSV).exists():
    gqa_sum = pd.read_csv(GQA_SUMMARY_CSV)
    # question_only = language prior floor
    qo_row = gqa_sum[gqa_sum["method"] == "question_only"]
    if not qo_row.empty:
        comparison_rows.append({
            "model":           "Question-Only (no scene)",
            "exact_accuracy":  qo_row["exact_accuracy"].values[0],
            "lenient_accuracy":qo_row["lenient_accuracy"].values[0],
            "avg_latency_ms":  qo_row["avg_latency_ms"].values[0],
            "visual_tokens":   0,
            "type":            "language prior",
        })
    # Best SST method by exact accuracy (excluding question_only)
    sst_rows = gqa_sum[gqa_sum["method"] != "question_only"]
    if not sst_rows.empty:
        best = sst_rows.loc[sst_rows["exact_accuracy"].idxmax()]
        comparison_rows.append({
            "model":           f"Best SST ({best['method_name']})",
            "exact_accuracy":  best["exact_accuracy"],
            "lenient_accuracy":best["lenient_accuracy"],
            "avg_latency_ms":  best["avg_latency_ms"],
            "visual_tokens":   best["avg_tokens"],
            "type":            "SST (text-only)",
        })

comp_df = pd.DataFrame(comparison_rows).sort_values("exact_accuracy", ascending=False)
comp_df["exact_accuracy_pct"]   = (comp_df["exact_accuracy"]   * 100).round(2)
comp_df["lenient_accuracy_pct"] = (comp_df["lenient_accuracy"] * 100).round(2)
comp_df["latency_ms"]           = comp_df["avg_latency_ms"].round(0)

print("\n── 3-Way Comparison ──────────────────────────────────────────────────────")
print(comp_df[["model","exact_accuracy_pct","lenient_accuracy_pct",
               "latency_ms","visual_tokens","type"]].to_string(index=False))

comp_df.to_csv(RESULTS_DIR / "blip2_llava_sst_comparison.csv", index=False)
print(f"\nTable saved → {RESULTS_DIR}/blip2_llava_sst_comparison.csv")


In [ ]:
# ── Comparison plot ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("BLIP-2 vs LLaVA vs SST — GQA Comparison", fontsize=13, y=1.02)

COLOR_MAP = {
    "VLM baseline":  "#3498DB",
    "SST (text-only)":"#27AE60",
    "language prior":"#95A5A6",
}
colors = [COLOR_MAP.get(t, "#BDC3C7") for t in comp_df["type"]]
models = comp_df["model"].tolist()
x      = range(len(models))

# ── Plot 1: Exact accuracy ────────────────────────────────────────────────
ax = axes[0]
bars = ax.bar(x, comp_df["exact_accuracy_pct"], color=colors, edgecolor="white")
for bar, val in zip(bars, comp_df["exact_accuracy_pct"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f"{val:.1f}%", ha="center", fontsize=9, fontweight="bold")
ax.set_xticks(list(x))
ax.set_xticklabels(models, rotation=25, ha="right", fontsize=8)
ax.set_ylabel("Exact accuracy (%)", fontsize=10)
ax.set_title("Exact accuracy", fontsize=11)
ax.grid(axis="y", alpha=0.3)

# ── Plot 2: Latency ───────────────────────────────────────────────────────
ax2 = axes[1]
bars2 = ax2.bar(x, comp_df["latency_ms"], color=colors, edgecolor="white")
for bar, val in zip(bars2, comp_df["latency_ms"]):
    if not pd.isna(val):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                 f"{val:.0f}ms", ha="center", fontsize=9, fontweight="bold")
ax2.set_xticks(list(x))
ax2.set_xticklabels(models, rotation=25, ha="right", fontsize=8)
ax2.set_ylabel("Avg latency (ms)", fontsize=10)
ax2.set_title("Inference latency", fontsize=11)
ax2.grid(axis="y", alpha=0.3)

# ── Plot 3: Accuracy vs latency scatter ───────────────────────────────────
ax3 = axes[2]
for i, row in comp_df.iterrows():
    if pd.isna(row["latency_ms"]): continue
    ax3.scatter(row["latency_ms"], row["exact_accuracy_pct"],
                color=COLOR_MAP.get(row["type"], "#BDC3C7"), s=120, zorder=3)
    ax3.annotate(row["model"], (row["latency_ms"], row["exact_accuracy_pct"]),
                 fontsize=8, textcoords="offset points", xytext=(6, 2))
ax3.set_xlabel("Avg latency (ms)", fontsize=10)
ax3.set_ylabel("Exact accuracy (%)", fontsize=10)
ax3.set_title("Accuracy vs latency", fontsize=11)
ax3.grid(alpha=0.3)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(color=c, label=t) for t, c in COLOR_MAP.items()]
fig.legend(handles=legend_elements, loc="lower center", ncol=3,
           fontsize=9, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
fig_path = RESULTS_DIR / "blip2_llava_sst_comparison.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {fig_path}")


In [ ]:
# ── BLIP-2 error analysis: where does it fail vs LLaVA? ──────────────────
# Tags each question with an error type and shows per-type accuracy.
# Mirrors the analysis in sst_eval_main so results are directly comparable.

def tag_error_type(question: str) -> str:
    q = question.lower()
    if any(w in q for w in ["color", "colour"]):          return "color"
    if any(w in q for w in ["how many", "count", "number of"]): return "counting"
    if any(w in q for w in [" left"," right","above","below",
                              "next to","behind","in front","between",
                              "beside","near","under","over"]): return "spatial"
    if any(w in q for w in ["is there","are there","does the",
                              "any ","exist"]):             return "existence"
    if any(w in q for w in ["what size","how big","how tall",
                              "what shape","what type","what kind",
                              "what material"]):            return "attribute"
    if any(w in q for w in ["doing","wearing","holding",
                              "riding","carrying","eating"]): return "action"
    if q.startswith("what is") or q.startswith("what are"): return "object_id"
    return "other"

results_df["error_type"] = results_df["question"].apply(tag_error_type)

print("── BLIP-2 accuracy by error type ────────────────────────────────────────")
et = (results_df.groupby("error_type")
      .agg(n=("exact_correct","count"),
           exact_acc=("exact_correct","mean"),
           lenient_acc=("lenient_correct","mean"))
      .reset_index()
      .sort_values("exact_acc", ascending=False))
et["exact_acc_pct"]   = (et["exact_acc"]   * 100).round(1)
et["lenient_acc_pct"] = (et["lenient_acc"] * 100).round(1)
print(et[["error_type","n","exact_acc_pct","lenient_acc_pct"]].to_string(index=False))

et_path = RESULTS_DIR / f"{PREFIX}_error_type_breakdown.csv"
et.to_csv(et_path, index=False)
print(f"\nError type breakdown saved → {et_path}")
